In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# ====================================================
# Task 1: Dataset Understanding
# ====================================================
# Load dataset
df = pd.read_csv("customer_support_text_classification.csv")

print("====================================================")
print("     TASK 1: DATASET UNDERSTANDING REPORT")
print("====================================================")
print(f"Total Number of Records: {len(df)}")
print(f"Target Labels/Classes:  {df['sentiment_label'].unique()}")
print(f"Average Text Length:     {df['word_count'].mean():.2f} words")
print("\nClass Distribution:")
print(df['sentiment_label'].value_counts())

print("\nSample Text Records:")
print(df[['customer_message', 'sentiment_label']].head(3))
print("-" * 52)


# ====================================================
# Task 2: Text Preprocessing
# ====================================================
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()                                 # Lowercasing
    text = re.sub(r'[^a-z0-9\s]', '', text)             # Remove special characters
    text = " ".join(text.split())                       # Clean up extra whitespaces
    return text

df['cleaned_message'] = df['customer_message'].apply(clean_text)

# Encode classes into numerical integers (0, 1, 2)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment_label'])

# Split into Train and Test splits (80/20) with stratification
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)


# ====================================================
# Task 3 & 4: Text Vectorization & Baseline Model
# ====================================================
# Vectorization using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=2000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

# Training Baseline Model (Logistic Regression)
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_tfidf, y_train)

# Evaluation
baseline_preds = baseline_model.predict(X_test_tfidf)

print("\n====================================================")
print("     TASK 4: TF-IDF + LOGISTIC REGRESSION BASELINE")
print("====================================================")
print(classification_report(y_test, baseline_preds, target_names=label_encoder.classes_))
print("-" * 52)


# ====================================================
# Task 5: Sequence Model (LSTM Implementation)
# ====================================================
# Text Vectorization using Tokenizer-based sequences
MAX_VOCAB_SIZE = 5000
MAX_SEQUENCE_LENGTH = 30

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_raw)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=MAX_SEQUENCE_LENGTH, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=MAX_SEQUENCE_LENGTH, padding='post')

num_classes = len(label_encoder.classes_)

# Building the LSTM Architecture
lstm_model = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("\n====================================================")
print("     TASK 5: LSTM ARCHITECTURE SUMMARY")
print("====================================================")
lstm_model.summary()

print("\nTraining LSTM Model for 5 Epochs...")
lstm_model.fit(X_train_seq, y_train, epochs=5, batch_size=32, validation_data=(X_test_seq, y_test), verbose=1)

# Generate predictions for the text sample display
lstm_preds = np.argmax(lstm_model.predict(X_test_seq), axis=-1)

# Dynamically determine the available sample print size (up to 5) to prevent IndexError
num_samples_available = min(5, len(X_test_raw))

sample_texts = X_test_raw.iloc[:num_samples_available].values
sample_true_labels = label_encoder.inverse_transform(y_test.iloc[:num_samples_available].values)
sample_pred_labels = label_encoder.inverse_transform(lstm_preds[:num_samples_available])

print("\n====================================================")
print("     LSTM MODEL SAMPLE PREDICTIONS OUTPUT")
print("====================================================")
for i in range(num_samples_available):
    print(f"Sample #{i+1}")
    print(f"Text:       {sample_texts[i]}")
    print(f"True Class: {sample_true_labels[i]}")
    print(f"Predicted:  {sample_pred_labels[i]}")
    print("-" * 52)

     TASK 1: DATASET UNDERSTANDING REPORT
Total Number of Records: 20
Target Labels/Classes:  ['neutral' 'positive' 'negative']
Average Text Length:     13.25 words

Class Distribution:
sentiment_label
neutral     7
positive    7
negative    6
Name: count, dtype: int64

Sample Text Records:
                                    customer_message sentiment_label
0  I need information about the payment process. ...         neutral
1      I need information about the payment process.         neutral
2  The refund process was fast and convenient. I ...        positive
----------------------------------------------------

     TASK 4: TF-IDF + LOGISTIC REGRESSION BASELINE
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67         1
     neutral       1.00      0.50      0.67         2
    positive       1.00      1.00      1.00         1

    accuracy                           0.75         4
   macro avg       0.83      0.83      0.78         4
w

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training LSTM Model for 5 Epochs...
Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.4375 - loss: 1.0998 - val_accuracy: 0.2500 - val_loss: 1.0985
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.1875 - loss: 1.0991 - val_accuracy: 0.2500 - val_loss: 1.1006
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - accuracy: 0.3125 - loss: 1.0958 - val_accuracy: 0.2500 - val_loss: 1.1025
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.3750 - loss: 1.0979 - val_accuracy: 0.2500 - val_loss: 1.1033
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.3750 - loss: 1.0967 - val_accuracy: 0.2500 - val_loss: 1.1042
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 476ms/step

     LSTM MODEL SAMPLE PREDICTIONS OUTPUT
Sample #1
Text:       can someone explain the difference between the plans please respond as soon as possible
True Class: neutral
Predicted:  positive
----------------------------------------------------
Sample #2
Text:       billing is incorrect and i was ch

Task 6: Attention and Transformer Reflection
1. Why RNNs Struggle with Long-Term Dependencies

As sequences grow long, standard Recurrent Neural Networks (RNNs) struggle due to the Vanishing Gradient Problem. During backpropagation through time (BPTT), errors are multiplied repeatedly by the recurrent weight matrices. If these weights are small, the gradients exponentially decay toward zero. This causes historical sequence steps to lose their influence, preventing the model from linking contextual clues separated by long text spans.

2. How LSTMs Help with Memory

Long Short-Term Memory (LSTM) cells solve this vulnerability by introducing an internal Cell State (C
t
​
 ), which acts as an information highway. This highway is regulated by three specialized gating mechanisms:

Forget Gate: Decides what information from historical states to discard.

Input Gate: Determines which new incoming structural details to store in the cell state.

Output Gate: Controls what information from the current cell state is emitted to the next hidden layer.

Because elements are adjusted primarily through linear additions rather than continuous multiplications, gradients can flow backward through time without vanishing completely.

3. What Attention Solves in Sequence-to-Sequence Tasks

Traditional Encoder-Decoder setups force an entire input sequence to compress into a single, fixed-length context vector. This creates an architectural bottleneck: long sentences lose critical context. Attention removes this bottleneck by allowing the decoder to look back at the entire input sequence at every output step. It calculates dynamic alignment scores, allowing the network to focus on specific, relevant input tokens regardless of their distance from the current generation step.

4. Why Transformers dominate Modern NLP & GenAI

Transformers changed NLP by replacing recurrence entirely with Self-Attention. This offers two major advantages:

Massive Parallelization: Because tokens are processed simultaneously rather than sequentially, models train much faster on modern hardware (GPUs) compared to RNNs or LSTMs.

Global Context: Self-attention measures the relationship between every pair of words in a sentence instantly, capturing long-range contextual relationships perfectly.

This highly scalable architecture serves as the foundation for modern Large Language Models (LLMs) like GPT, Claude, and Gemini.